# Full Project Notebook: e-SNLI to GMEG-EXP
This notebook is fully self-contained and runs the entire pipeline without importing `src/*.py`.


## Step Names and Datasets
Select old baseline explanation dataset - e-SNLI  
Extract old human explanations - e-SNLI  
Generate synthetic old LLM explanations - e-SNLI  
Build human vs LLM explanation training set - e-SNLI  
Train explanation-style detector - e-SNLI  
Run length-matched and style-only controls - e-SNLI  
Download and prepare LLM-era target explanations - GMEG-EXP  
Score newer human explanations with the detector - GMEG-EXP  
Compare old-human, old-LLM, and new-human score distributions - e-SNLI + GMEG-EXP  
Perform qualitative audit of high-score and low-score target explanations - GMEG-EXP  
Report detector performance and LLM-likeness shift - e-SNLI + GMEG-EXP


In [ ]:
# Install once in notebook if needed:
# %pip install -q pandas numpy scikit-learn matplotlib seaborn datasets openai tqdm joblib

from __future__ import annotations

import json
import os
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from datasets import load_dataset
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm


In [ ]:
@dataclass
class Config:
    project_root: Path = Path.cwd()
    data_raw_dir: Path = project_root / 'data' / 'raw'
    data_processed_dir: Path = project_root / 'data' / 'processed'
    outputs_dir: Path = project_root / 'outputs'
    models_dir: Path = outputs_dir / 'models'
    figures_dir: Path = outputs_dir / 'figures'
    reports_dir: Path = outputs_dir / 'reports'

    esnli_subset_size: int = 8000
    random_seed: int = 42
    test_size: float = 0.2

    llm_model: str = 'gpt-4o-mini'
    max_tokens: int = 140
    temperature: float = 0.2

    tfidf_max_features: int = 30000
    llm_like_threshold: float = 0.8
    length_q_low: float = 0.1
    length_q_high: float = 0.9

CFG = Config()

for p in [
    CFG.data_raw_dir,
    CFG.data_processed_dir,
    CFG.outputs_dir,
    CFG.models_dir,
    CFG.figures_dir,
    CFG.reports_dir,
]:
    p.mkdir(parents=True, exist_ok=True)

LABEL_MAP = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}


In [ ]:
_WORD_RE = re.compile(r'\b\w+\b')
STOPWORDS = {
    'a','an','and','are','as','at','be','by','for','from','has','he','in','is','it','its','of','on',
    'that','the','to','was','were','will','with','this','these','those','their','there','because',
    'but','or','if','then','than'
}


def _pick_first_existing(columns: list[str], candidates: list[str]) -> Optional[str]:
    cols = set(columns)
    for c in candidates:
        if c in cols:
            return c
    return None


def _tokenize(text: str) -> list[str]:
    return _WORD_RE.findall(str(text).lower())


def token_count(text: str) -> int:
    return len(_tokenize(text))


def style_feature_df(texts: pd.Series) -> pd.DataFrame:
    rows = []
    for txt in texts.astype(str).fillna(''):
        toks = _tokenize(txt)
        n = len(toks)
        uniq = len(set(toks))
        stop = sum(1 for t in toks if t in STOPWORDS)
        rows.append({
            'token_count': n,
            'char_count': len(txt),
            'punct_count': sum(ch in '.,!?:;' for ch in txt),
            'repeat_rate': 0.0 if n == 0 else 1.0 - (uniq / n),
            'stopword_ratio': 0.0 if n == 0 else stop / n,
            'lexical_diversity': 0.0 if n == 0 else uniq / n,
        })
    return pd.DataFrame(rows)


def length_filter(df: pd.DataFrame, col: str, q_low: float, q_high: float) -> pd.DataFrame:
    lengths = df[col].astype(str).map(token_count)
    lo = np.quantile(lengths, q_low)
    hi = np.quantile(lengths, q_high)
    return df[(lengths >= lo) & (lengths <= hi)].copy()


In [ ]:
def load_esnli_subset(n: int, seed: int) -> pd.DataFrame:
    ds = load_dataset('esnli')
    df = ds['train'].to_pandas()

    exp_col = _pick_first_existing(df.columns.tolist(), ['explanation_1', 'explanation', 'sentence_explanation'])
    if exp_col is None:
        raise ValueError('Could not find e-SNLI explanation column.')

    needed = ['premise', 'hypothesis', 'label', exp_col]
    out = (
        df[needed]
        .rename(columns={exp_col: 'explanation'})
        .dropna(subset=['premise', 'hypothesis', 'explanation'])
        .sample(n=min(n, len(df)), random_state=seed)
        .reset_index(drop=True)
    )
    out['label_name'] = out['label'].map(lambda x: LABEL_MAP.get(int(x), str(x)))
    return out


def load_gmeg_human_explanations(gmeg_root: Path) -> pd.DataFrame:
    if not gmeg_root.exists():
        raise FileNotFoundError(f'GMEG-EXP path not found: {gmeg_root}')

    files = list(gmeg_root.rglob('*.csv')) + list(gmeg_root.rglob('*.jsonl')) + list(gmeg_root.rglob('*.json'))
    if not files:
        raise FileNotFoundError('No CSV/JSON/JSONL files found under GMEG-EXP root.')

    frames = []
    for f in files:
        try:
            if f.suffix == '.csv':
                df = pd.read_csv(f)
            elif f.suffix == '.jsonl':
                df = pd.read_json(f, lines=True)
            else:
                df = pd.read_json(f)
        except Exception:
            continue

        if df.empty:
            continue

        cols = [c.lower() for c in df.columns]
        cmap = {c.lower(): c for c in df.columns}

        exp_col = _pick_first_existing(cols, ['explanation', 'human_explanation', 'rationale', 'reason', 'justification'])
        if exp_col is None:
            continue

        src_col = _pick_first_existing(cols, ['source', 'author', 'origin', 'generator', 'type', 'label'])

        cur = pd.DataFrame({'explanation': df[cmap[exp_col]]})
        if src_col:
            src = df[cmap[src_col]].astype(str).str.lower()
            human_mask = src.str.contains('human') | src.str.contains('annotator')
            cur = cur[human_mask]

        cur['source_file'] = str(f)
        frames.append(cur)

    if not frames:
        raise ValueError('Could not auto-detect human explanations in GMEG-EXP. Adjust column rules in this cell.')

    out = pd.concat(frames, ignore_index=True).dropna(subset=['explanation'])
    out['explanation'] = out['explanation'].astype(str).str.strip()
    out = out[out['explanation'].str.len() > 0].reset_index(drop=True)
    return out


In [ ]:
def build_esnli_prompt(premise: str, hypothesis: str, label: str) -> str:
    return (
        f'Premise: {premise}\n'
        f'Hypothesis: {hypothesis}\n'
        f'Gold label: {label}\n'
        'Write a 1-2 sentence explanation that justifies the gold label '
        'using only information from the premise and hypothesis.'
    )


def generate_explanations(prompts: list[str], model: str, max_tokens: int, temperature: float, sleep_s: float = 0.0) -> list[str]:
    api_key = os.getenv('OPENAI_API_KEY')
    if not api_key:
        raise EnvironmentError('OPENAI_API_KEY is not set.')

    client = OpenAI(api_key=api_key)
    outputs = []

    for p in tqdm(prompts, desc='Generating synthetic e-SNLI LLM explanations'):
        resp = client.responses.create(
            model=model,
            input=[
                {'role': 'system', 'content': 'You provide concise NLI explanations.'},
                {'role': 'user', 'content': p},
            ],
            max_output_tokens=max_tokens,
            temperature=temperature,
        )
        outputs.append((resp.output_text or '').strip())
        if sleep_s > 0:
            time.sleep(sleep_s)

    return outputs


In [ ]:
def train_detector(old_human: pd.DataFrame, old_llm: pd.DataFrame):
    df = pd.concat([
        old_human[['explanation']].assign(y=0),
        old_llm[['explanation']].assign(y=1),
    ], ignore_index=True)

    X_train, X_test, y_train, y_test = train_test_split(
        df['explanation'].astype(str),
        df['y'],
        test_size=CFG.test_size,
        random_state=CFG.random_seed,
        stratify=df['y'],
    )

    vectorizer = TfidfVectorizer(max_features=CFG.tfidf_max_features, ngram_range=(1, 2), lowercase=True)
    Xtr = vectorizer.fit_transform(X_train)
    Xte = vectorizer.transform(X_test)

    clf = LogisticRegression(max_iter=2000)
    clf.fit(Xtr, y_train)

    p_test = clf.predict_proba(Xte)[:, 1]
    y_pred = (p_test >= 0.5).astype(int)
    metrics = {
        'f1': float(f1_score(y_test, y_pred)),
        'auroc': float(roc_auc_score(y_test, p_test)),
    }
    return vectorizer, clf, metrics


def run_controls(old_human: pd.DataFrame, old_llm: pd.DataFrame):
    combined = pd.concat([
        old_human[['explanation']].assign(y=0),
        old_llm[['explanation']].assign(y=1),
    ], ignore_index=True)

    length_df = length_filter(combined, 'explanation', CFG.length_q_low, CFG.length_q_high)
    X_train, X_test, y_train, y_test = train_test_split(
        length_df['explanation'].astype(str),
        length_df['y'],
        test_size=CFG.test_size,
        random_state=CFG.random_seed,
        stratify=length_df['y'],
    )

    vec = TfidfVectorizer(max_features=CFG.tfidf_max_features, ngram_range=(1, 2))
    Xtr = vec.fit_transform(X_train)
    Xte = vec.transform(X_test)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(Xtr, y_train)
    p = clf.predict_proba(Xte)[:, 1]
    yhat = (p >= 0.5).astype(int)

    length_metrics = {
        'f1': float(f1_score(y_test, yhat)),
        'auroc': float(roc_auc_score(y_test, p)),
        'n': int(len(length_df)),
    }

    X_style = style_feature_df(combined['explanation'])
    y_style = combined['y'].values
    Xs_train, Xs_test, ys_train, ys_test = train_test_split(
        X_style, y_style,
        test_size=CFG.test_size,
        random_state=CFG.random_seed,
        stratify=y_style,
    )

    style_clf = LogisticRegression(max_iter=2000)
    style_clf.fit(Xs_train, ys_train)
    ps = style_clf.predict_proba(Xs_test)[:, 1]
    yhs = (ps >= 0.5).astype(int)

    style_metrics = {
        'f1': float(f1_score(ys_test, yhs)),
        'auroc': float(roc_auc_score(ys_test, ps)),
    }

    return {'length_matched': length_metrics, 'style_only': style_metrics}


def score_groups(vectorizer, clf, old_human: pd.DataFrame, old_llm: pd.DataFrame, gmeg_human: pd.DataFrame):
    groups = [
        old_human[['explanation']].assign(group='old_human'),
        old_llm[['explanation']].assign(group='old_llm'),
        gmeg_human[['explanation']].assign(group='new_human'),
    ]
    scored = []
    for g in groups:
        X = vectorizer.transform(g['explanation'].astype(str))
        p = clf.predict_proba(X)[:, 1]
        cur = g.copy()
        cur['llm_likeness'] = p
        scored.append(cur)
    return pd.concat(scored, ignore_index=True)


def summarize_scores(scored: pd.DataFrame) -> pd.DataFrame:
    return (
        scored.groupby('group')['llm_likeness']
        .agg(
            mean='mean',
            median='median',
            pct_above_0_8=lambda s: (s >= CFG.llm_like_threshold).mean() * 100,
            n='count',
        )
        .reset_index()
    )


def qualitative_audit(scored: pd.DataFrame, n_each: int = 20) -> pd.DataFrame:
    target = scored[scored['group'] == 'new_human'].copy()
    high = target.nlargest(n_each, 'llm_likeness').assign(bucket='high_score')
    low = target.nsmallest(n_each, 'llm_likeness').assign(bucket='low_score')
    return pd.concat([high, low], ignore_index=True)


def plot_distributions(scored: pd.DataFrame, output_png: Path) -> None:
    sns.set_theme(style='whitegrid')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    sns.histplot(
        data=scored,
        x='llm_likeness',
        hue='group',
        bins=30,
        stat='density',
        common_norm=False,
        alpha=0.4,
        ax=axes[0],
    )
    axes[0].set_title('LLM-likeness score distributions')
    axes[0].set_xlim(0, 1)

    sns.boxplot(data=scored, x='group', y='llm_likeness', ax=axes[1])
    axes[1].set_title('Group score comparison')
    axes[1].set_ylim(0, 1)

    plt.tight_layout()
    fig.savefig(output_png, dpi=180)
    plt.close(fig)


In [ ]:
# --------------------
# Run entire pipeline
# --------------------

GMEG_ROOT = Path('data/raw/GMEG-EXP')
GENERATE_LLM = True  # Set False after first run if cached synthetic explanations exist

esnli_cache = CFG.data_processed_dir / 'esnli_old_human.csv'
llm_cache = CFG.data_processed_dir / 'esnli_old_llm.csv'

if esnli_cache.exists():
    old_human = pd.read_csv(esnli_cache)
else:
    old_human = load_esnli_subset(CFG.esnli_subset_size, CFG.random_seed)
    old_human.to_csv(esnli_cache, index=False)

if llm_cache.exists():
    old_llm = pd.read_csv(llm_cache)
else:
    if not GENERATE_LLM:
        raise FileNotFoundError(f'Missing {llm_cache}. Set GENERATE_LLM=True for first run.')

    prompts = [
        build_esnli_prompt(p, h, LABEL_MAP.get(int(l), str(l)))
        for p, h, l in zip(old_human['premise'], old_human['hypothesis'], old_human['label'])
    ]
    llm_explanations = generate_explanations(
        prompts,
        model=CFG.llm_model,
        max_tokens=CFG.max_tokens,
        temperature=CFG.temperature,
    )
    old_llm = old_human[['premise', 'hypothesis', 'label', 'label_name']].copy()
    old_llm['explanation'] = llm_explanations
    old_llm.to_csv(llm_cache, index=False)

old_human = old_human.dropna(subset=['explanation']).copy()
old_llm = old_llm.dropna(subset=['explanation']).copy()

vectorizer, clf, metrics = train_detector(old_human, old_llm)
controls = run_controls(old_human, old_llm)

gmeg_human = load_gmeg_human_explanations(GMEG_ROOT)

scored = score_groups(vectorizer, clf, old_human, old_llm, gmeg_human)
summary = summarize_scores(scored)
audit = qualitative_audit(scored, n_each=20)

scored.to_csv(CFG.outputs_dir / 'scored_explanations.csv', index=False)
summary.to_csv(CFG.outputs_dir / 'score_summary.csv', index=False)
audit.to_csv(CFG.outputs_dir / 'qualitative_audit_samples.csv', index=False)
joblib.dump({'vectorizer': vectorizer, 'clf': clf, 'metrics': metrics, 'controls': controls}, CFG.models_dir / 'esnli_llm_detector.joblib')

fig_path = CFG.figures_dir / 'llm_likeness_distributions.png'
plot_distributions(scored, fig_path)

report_lines = [
    '# Explanation-Style Shift Report',
    '',
    '## Detector Performance (e-SNLI split)',
    f"- F1: {metrics['f1']:.4f}",
    f"- AUROC: {metrics['auroc']:.4f}",
    '',
    '## Controls',
    f"- Length-matched TF-IDF+LR: F1={controls['length_matched']['f1']:.4f}, AUROC={controls['length_matched']['auroc']:.4f}, n={controls['length_matched']['n']}",
    f"- Style-only LR: F1={controls['style_only']['f1']:.4f}, AUROC={controls['style_only']['auroc']:.4f}",
    '',
    '## LLM-likeness Distribution Summary',
]
for _, row in summary.iterrows():
    report_lines.append(
        f"- {row['group']}: mean={row['mean']:.4f}, median={row['median']:.4f}, pct>=0.8={row['pct_above_0_8']:.2f}%, n={int(row['n'])}"
    )
(CFG.reports_dir / 'final_report.md').write_text('\n'.join(report_lines), encoding='utf-8')

manifest = {
    'metrics': metrics,
    'controls': controls,
    'files': {
        'scored_csv': str(CFG.outputs_dir / 'scored_explanations.csv'),
        'summary_csv': str(CFG.outputs_dir / 'score_summary.csv'),
        'audit_csv': str(CFG.outputs_dir / 'qualitative_audit_samples.csv'),
        'model': str(CFG.models_dir / 'esnli_llm_detector.joblib'),
        'figure': str(fig_path),
        'report': str(CFG.reports_dir / 'final_report.md'),
    }
}
(CFG.outputs_dir / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print('Pipeline completed successfully.')
print('Metrics:', metrics)
summary


In [ ]:
from IPython.display import Image, display

display(Image(filename='outputs/figures/llm_likeness_distributions.png'))


In [ ]:
audit = pd.read_csv('outputs/qualitative_audit_samples.csv')
audit.head(20)
